In [1]:
# ── STEP 1: Install dependencies ─────────────────────────────
!pip install openpyxl xlsxwriter -q

# ── STEP 2: Mount Google Drive ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.3 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
"""
============================================================
  BankNifty Short Strangle Backtest
============================================================
Strategy  : Short Strangle (Sell CE + Sell PE)
Universe  : Week-1 trading days only (Mon → first Wednesday)
Entry     : 09:20 candle close, strike closest to Rs.50 premium
Exit      : 15:20 candle close OR 50% SL on entry price (whichever first)
SL check  : Uses High column of each 1-min bar
Quantity  : 1 lot = 15 units per leg, per day (no compounding)
Expiry    : First Wednesday of each month (Week-1)



"""

import time, warnings
import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot   as plt
import matplotlib.ticker   as mticker
from io              import BytesIO
from openpyxl        import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side)
from openpyxl.utils  import get_column_letter
from openpyxl.drawing.image import Image as XLImage

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════
#  CONFIGURATION  ← edit these two paths before running
# ════════════════════════════════════════════════════════════
OPTIONS_PATH    = '/content/drive/MyDrive/Options_data_2023.parquet'      # CSV or .parquet
SPOT_PATH       = '/content/BANKNIFTY_SPOT.csv'
OUTPUT_PATH     = 'BankNifty_ShortStrangle_Backtest.xlsx'

INITIAL_CAPITAL = 1_000_000          # Rs. 10,00,000
STARTING_NAV    = 100.0
LOT_SIZE        = 15
NUM_LOTS        = 1
QUANTITY        = LOT_SIZE * NUM_LOTS   # 15 per leg

TARGET_PREM     = 50.0               # select strike closest to Rs.50
SL_PCT          = 0.50               # 50% stop-loss on entry price
ENTRY_TIME      = '09:20'
EXIT_TIME       = '15:20'

STEP_TIMES: dict = {}


# ════════════════════════════════════════════════════════════
#  STEP 1 — LOAD OPTIONS DATA
# ════════════════════════════════════════════════════════════
def load_options(path: str) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 1]  Loading options data ...")

    if path.endswith('.parquet'):
        df = pd.read_parquet(path)
        df.columns = [c.strip() for c in df.columns]
    else:
        df = pd.read_csv(
            path,
            usecols=['Date', 'Ticker', 'Time', 'High', 'Close', 'Call/Put'],
            dtype={'Time': str},
            low_memory=False,
            encoding='latin1',
        )
        df.columns = [c.strip() for c in df.columns]

    if 'Call/Put' in df.columns:
        df.rename(columns={'Call/Put': 'OptionType'}, inplace=True)

    # Normalise types — plain str for Ticker avoids all category/str merge bugs
    df['Date']       = pd.to_datetime(df['Date']).dt.date
    df['Time']       = df['Time'].astype(str).str[:5]
    df['Ticker']     = df['Ticker'].astype(str)

    ticker_str       = df['Ticker']
    df['Strike']     = pd.to_numeric(
        ticker_str.str.extract(r'BANKNIFTY(\d+)(?:CE|PE)', expand=False),
        errors='coerce')

    if 'OptionType' not in df.columns or df['OptionType'].isna().all():
        df['OptionType'] = ticker_str.str.extract(r'BANKNIFTY\d+(CE|PE)', expand=False)
    df['OptionType'] = df['OptionType'].astype(str).str.strip().str.upper()

    elapsed = time.time() - t0
    STEP_TIMES['1_load_options'] = elapsed
    print(f"          Rows: {len(df):,}   Columns: {list(df.columns)}")
    print(f"          Time: {elapsed:.1f}s")
    return df


# ════════════════════════════════════════════════════════════
#  STEP 2 — LOAD SPOT DATA
# ════════════════════════════════════════════════════════════
def load_spot(path: str) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 2]  Loading BankNifty spot data ...")

    spot = pd.read_csv(path, low_memory=False)
    spot.columns = [c.strip().lower() for c in spot.columns]

    # Auto-detect datetime and close columns (handles ts/c or datetime/close)
    if 'ts' in spot.columns:
        spot.rename(columns={'ts': '_dt', 'c': '_close'}, inplace=True)
    else:
        dt_col    = next((c for c in spot.columns
                          if any(k in c for k in ('date','time','dt','ts'))), spot.columns[0])
        close_col = next((c for c in spot.columns
                          if c in ('close','c','ltp','last')), spot.columns[-1])
        spot.rename(columns={dt_col: '_dt', close_col: '_close'}, inplace=True)

    spot['_dt']    = pd.to_datetime(spot['_dt'])
    spot['Date']   = spot['_dt'].dt.date
    spot['Time']   = spot['_dt'].dt.strftime('%H:%M')
    spot['Spot_Close'] = pd.to_numeric(spot['_close'], errors='coerce')
    spot = spot[['Date', 'Time', 'Spot_Close']].copy()

    elapsed = time.time() - t0
    STEP_TIMES['2_load_spot'] = elapsed
    print(f"          Rows: {len(spot):,}   Dates: {spot['Date'].nunique()}")
    print(f"          Time: {elapsed:.1f}s")
    return spot


# ════════════════════════════════════════════════════════════
#  STEP 3 — IDENTIFY WEEK-1 TRADING DATES
#  Week 1 = all trading days from Monday up to and including
#  the FIRST Wednesday of each calendar month (= expiry day)
# ════════════════════════════════════════════════════════════
def get_week1_dates(all_dates) -> list:
    t0 = time.time()
    print("[STEP 3]  Identifying Week-1 trading dates ...")

    ds    = pd.to_datetime(pd.Series(sorted(set(all_dates))))
    week1 = []

    for (y, m), grp in ds.groupby([ds.dt.year, ds.dt.month]):
        weds = grp[grp.dt.dayofweek == 2]          # Wednesday = 2
        if weds.empty:
            continue
        first_wed = weds.iloc[0]
        iso_week  = first_wed.isocalendar()[1]
        iso_year  = first_wed.isocalendar()[0]
        # All trading days in the same ISO week up to first_wed
        same_wk = grp[
            (grp.dt.isocalendar().week.values == iso_week) &
            (grp.dt.isocalendar().year.values == iso_year) &
            (grp <= first_wed)
        ]
        week1.extend(same_wk.dt.date.tolist())

    week1   = sorted(set(week1))
    expiry  = [d for d in week1 if pd.Timestamp(d).dayofweek == 2]
    elapsed = time.time() - t0
    STEP_TIMES['3_week1_dates'] = elapsed
    print(f"          Week-1 trading days : {len(week1)}")
    print(f"          Expiry days (Wed)   : {len(expiry)}")
    print(f"          Time: {elapsed:.2f}s")
    return week1


# ════════════════════════════════════════════════════════════
#  STEP 4 — STRIKE SELECTION MODULE (vectorised)
#  At 09:20 on each Week-1 day:
#    → pick the CE strike whose Close is closest to Rs.50
#    → pick the PE strike whose Close is closest to Rs.50
# ════════════════════════════════════════════════════════════
def select_strikes(df: pd.DataFrame, week1_dates: list) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 4]  Strike selection (closest to Rs.50 at 09:20) ...")

    mask = (
        df['Time'].str.startswith(ENTRY_TIME) &
        df['Date'].isin(set(week1_dates)) &
        df['OptionType'].isin(['CE', 'PE'])
    )
    entry = df.loc[mask, ['Date', 'Ticker', 'Strike', 'OptionType', 'Close']].copy()
    entry['Dist'] = (entry['Close'] - TARGET_PREM).abs()

    # Vectorised idxmin per (Date, OptionType) — no loop
    idx      = entry.groupby(['Date', 'OptionType'], sort=False)['Dist'].idxmin()
    selected = entry.loc[idx].reset_index(drop=True)
    selected.rename(columns={'Close': 'EntryPrice', 'Ticker': 'EntryTicker'}, inplace=True)

    # SL_Level lives ONLY here — never propagated into intra bars (prevents NaN bug)
    selected['SL_Level'] = (selected['EntryPrice'] * (1.0 + SL_PCT)).round(2)

    elapsed = time.time() - t0
    STEP_TIMES['4_strike_selection'] = elapsed
    print(f"          Legs selected : {len(selected)}  ({len(selected)//2} days x 2 legs)")
    print(f"          Time: {elapsed:.2f}s")
    return selected


# ════════════════════════════════════════════════════════════
#  STEP 5 — SIGNAL GENERATION + EXIT LOGIC (fully vectorised)
#
#  Entry : 09:20 close
#  Exit  : 15:20 close  OR  first 1-min bar where High >= SL_Level
#  SL exit price = EntryPrice * 1.5  (canonical, from `selected`)
#
#  Design: SL_Level is attached to intra bars ONLY to detect
#  trigger; exit price is set from selected AFTER the flag, so
#  category/str dtype mismatches cannot corrupt the SL price.
# ════════════════════════════════════════════════════════════
def get_exits(df: pd.DataFrame, selected: pd.DataFrame) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 5]  Computing exits and stop-losses (vectorised) ...")

    traded_tickers = set(selected['EntryTicker'])
    traded_dates   = set(selected['Date'])

    # Intraday bars after entry time, for our traded legs only
    intra = df[
        (df['Time'] > ENTRY_TIME) &
        df['Date'].isin(traded_dates) &
        df['Ticker'].isin(traded_tickers)
    ][['Date', 'Ticker', 'Time', 'High', 'Close']].copy()

    # Attach SL_Level for trigger detection only
    sl_ref = (selected[['EntryTicker', 'SL_Level']]
              .rename(columns={'EntryTicker': 'Ticker'}))
    intra  = intra.merge(sl_ref, on='Ticker', how='left')

    # ── 1. First bar where High >= SL_Level ──────────────────
    sl_hit   = intra[intra['High'] >= intra['SL_Level']]
    sl_first = (
        sl_hit
        .sort_values('Time')
        .groupby(['Date', 'Ticker'], sort=False)
        .agg(SL_Time=('Time', 'first'))
        .reset_index()
    )

    # ── 2. Normal exit at 15:20 close ────────────────────────
    norm_exit = (
        intra[intra['Time'].str.startswith(EXIT_TIME)]
        .groupby(['Date', 'Ticker'], sort=False)
        .agg(Norm_Time=('Time', 'last'), Norm_Close=('Close', 'last'))
        .reset_index()
    )

    # ── 3. Fallback: last bar of the day ─────────────────────
    last_bar = (
        intra.sort_values('Time')
        .groupby(['Date', 'Ticker'], sort=False)
        .agg(Last_Time=('Time', 'last'), Last_Close=('Close', 'last'))
        .reset_index()
    )

    # ── 4. Merge everything onto selected ────────────────────
    res = selected.copy()
    res = res.merge(sl_first,  left_on=['Date','EntryTicker'],
                    right_on=['Date','Ticker'], how='left').drop(columns='Ticker')
    res = res.merge(norm_exit, left_on=['Date','EntryTicker'],
                    right_on=['Date','Ticker'], how='left').drop(columns='Ticker')
    res = res.merge(last_bar,  left_on=['Date','EntryTicker'],
                    right_on=['Date','Ticker'], how='left').drop(columns='Ticker')

    # ── 5. Resolve exit price & time ─────────────────────────
    res['SL_Hit'] = res['SL_Time'].notna()

    # SL exit price comes from SL_Level in `selected` (always correct)
    res['ExitPrice'] = np.where(
        res['SL_Hit'],
        res['SL_Level'],                                        # EntryPrice * 1.5
        np.where(
            res['Norm_Close'].notna(),
            res['Norm_Close'],                                  # normal 15:20 exit
            res['Last_Close'].fillna(res['EntryPrice'])         # fallback
        )
    ).astype(float).round(2)

    res['ExitTime'] = np.where(
        res['SL_Hit'],
        res['SL_Time'],
        np.where(
            res['Norm_Time'].notna(),
            res['Norm_Time'],
            res['Last_Time'].fillna(EXIT_TIME)
        )
    )

    res.drop(columns=['SL_Level','SL_Time','Norm_Time','Norm_Close',
                       'Last_Time','Last_Close'], inplace=True)

    elapsed = time.time() - t0
    STEP_TIMES['5_exit_logic'] = elapsed
    print(f"          SL triggered : {res['SL_Hit'].sum()} legs")
    print(f"          Time: {elapsed:.2f}s")
    return res


# ════════════════════════════════════════════════════════════
#  STEP 6 — POSITION SIZING + TRADE SHEET
# ════════════════════════════════════════════════════════════
def build_tradesheet(merged: pd.DataFrame, spot: pd.DataFrame,
                     week1_dates: list) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 6]  Building trade sheet ...")

    merged['Date'] = pd.to_datetime(merged['Date']).dt.date
    spot['Date']   = pd.to_datetime(spot['Date']).dt.date

    # Spot close at 09:20 — one clean row per day
    spot_entry = (spot[spot['Time'].str.startswith(ENTRY_TIME)]
                  [['Date','Spot_Close']]
                  .drop_duplicates('Date'))
    merged = merged.merge(spot_entry, on='Date', how='left')

    # Expiry flag (first Wednesday of month)
    expiry_dates       = {d for d in week1_dates if pd.Timestamp(d).dayofweek == 2}
    merged['IsExpiry'] = merged['Date'].isin(expiry_dates)

    # Position sizing — fixed, no compounding
    merged['Quantity']   = QUANTITY
    merged['EntryValue'] = (merged['EntryPrice'] * QUANTITY).round(2)
    merged['ExitValue']  = (merged['ExitPrice']  * QUANTITY).round(2)

    # P&L — short position: profit when price falls
    merged['GrossPnL'] = ((merged['EntryPrice'] - merged['ExitPrice']) * QUANTITY).round(2)
    merged['PnL_Pct']  = ((merged['EntryPrice'] - merged['ExitPrice'])
                           / merged['EntryPrice'] * 100).round(4)

    # Sort chronologically, CE before PE each day
    merged = (merged
              .sort_values(['Date','OptionType'], ascending=[True, False])
              .reset_index(drop=True))

    # Available capital — day-level (both legs combined before updating)
    day_pnl = (merged.groupby('Date')['GrossPnL']
                .sum().rename('DailyPnL').reset_index())
    day_pnl['CumPnL_Day'] = day_pnl['DailyPnL'].cumsum()
    day_pnl['AvailCap']   = (INITIAL_CAPITAL + day_pnl['CumPnL_Day']).round(2)
    merged = merged.merge(day_pnl[['Date','AvailCap']], on='Date', how='left')

    # Cumulative P&L trade-wise (for NAV equity curve)
    merged['CumPnL'] = merged['GrossPnL'].cumsum().round(2)
    merged['NAV']    = (STARTING_NAV
                        + merged['CumPnL'] / INITIAL_CAPITAL * STARTING_NAV
                        ).round(4)

    elapsed = time.time() - t0
    STEP_TIMES['6_tradesheet'] = elapsed
    print(f"          Rows built : {len(merged)}")
    print(f"          Time: {elapsed:.2f}s")
    return merged


# ════════════════════════════════════════════════════════════
#  STEP 7 — STATISTICAL ANALYSIS
# ════════════════════════════════════════════════════════════
def compute_stats(ts: pd.DataFrame) -> dict:
    t0 = time.time()
    print("[STEP 7]  Computing statistics ...")

    nav      = ts['NAV'].values
    n_days   = ts['Date'].nunique()
    years    = n_days / 252

    # CAGR
    cagr     = ((nav[-1] / STARTING_NAV) ** (1 / max(years, 1/252)) - 1) * 100

    # Max Drawdown (trade-wise from running NAV peak)
    run_peak = np.maximum.accumulate(nav)
    dd_arr   = (nav - run_peak) / run_peak * 100
    max_dd   = dd_arr.min()

    stats = {
        'CAGR_%'        : round(cagr,   2),
        'Max_Drawdown_%': round(max_dd, 2),
    }

    # Winner / loser counts
    for opt in ['CE', 'PE', 'Both']:
        sub = ts if opt == 'Both' else ts[ts['OptionType'] == opt]
        w = (sub['GrossPnL'] > 0).sum()
        l = (sub['GrossPnL'] <= 0).sum()
        t = len(sub)
        stats.update({
            f'{opt}_Winners' : int(w),
            f'{opt}_Losers'  : int(l),
            f'{opt}_Total'   : int(t),
            f'{opt}_Win_Pct' : round(w/t*100, 2) if t else 0,
            f'{opt}_Loss_Pct': round(l/t*100, 2) if t else 0,
        })

    # Average % P&L — expiry vs non-expiry, per option type
    for opt in ['CE', 'PE', 'Both']:
        sub = ts if opt == 'Both' else ts[ts['OptionType'] == opt]
        for flag, lbl in [(True,'Expiry'), (False,'NonExpiry')]:
            g = sub[sub['IsExpiry'] == flag]['PnL_Pct']
            stats[f'{opt}_{lbl}_Avg_PnL_%'] = round(g.mean(), 4) if len(g) else float('nan')

    # Monthly P&L table
    tc         = ts.copy()
    tc['Date'] = pd.to_datetime(tc['Date'])
    tc['YM']   = tc['Date'].dt.to_period('M')
    mnav       = tc.groupby('YM')['NAV'].last()
    mp         = pd.DataFrame({'EndNAV': mnav})
    mp['PrevNAV']       = mp['EndNAV'].shift(1).fillna(STARTING_NAV)
    mp['Monthly_PnL%']  = ((mp['EndNAV'] - mp['PrevNAV']) / mp['PrevNAV'] * 100).round(2)

    stats['Monthly_PnL_Table'] = mp
    stats['NAV_Series']        = pd.Series(nav)
    stats['DD_Series']         = pd.Series(dd_arr)

    elapsed = time.time() - t0
    STEP_TIMES['7_statistics'] = elapsed
    print(f"          CAGR             : {cagr:.2f}%")
    print(f"          Max Drawdown     : {max_dd:.2f}%")
    print(f"          Combined Win Rate: {stats['Both_Win_Pct']:.1f}%")
    print(f"          Time: {elapsed:.2f}s")
    return stats


# ════════════════════════════════════════════════════════════
#  STEP 8 — CHARTS
# ════════════════════════════════════════════════════════════
def make_charts(stats: dict) -> tuple:
    t0    = time.time()
    nav_v = stats['NAV_Series'].values
    dd_v  = stats['DD_Series'].values
    tn    = np.arange(1, len(nav_v) + 1)

    # ── Equity Curve ─────────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(14, 5))
    ax1.plot(tn, nav_v, color='#1a6496', linewidth=1.8, zorder=3, label='NAV')
    ax1.fill_between(tn, STARTING_NAV, nav_v,
                     where=(nav_v >= STARTING_NAV),
                     alpha=0.18, color='#27ae60', label='Above base')
    ax1.fill_between(tn, STARTING_NAV, nav_v,
                     where=(nav_v < STARTING_NAV),
                     alpha=0.18, color='#e74c3c', label='Below base')
    ax1.axhline(STARTING_NAV, color='grey', linestyle='--',
                linewidth=0.9, label=f'Base NAV = {STARTING_NAV}')
    # Annotate final NAV
    ax1.annotate(f"Final NAV: {nav_v[-1]:.4f}",
                 xy=(tn[-1], nav_v[-1]),
                 xytext=(-60, 12), textcoords='offset points',
                 fontsize=9, color='#1a6496',
                 arrowprops=dict(arrowstyle='->', color='#1a6496', lw=0.8))
    ax1.set_title('BankNifty Short Strangle — Equity Curve (Base NAV = 100)',
                  fontsize=13, fontweight='bold', pad=10)
    ax1.set_xlabel('Trade Number', fontsize=10)
    ax1.set_ylabel('NAV', fontsize=10)
    ax1.legend(fontsize=9, loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    plt.tight_layout()
    plt.show()
    buf1 = BytesIO()
    fig1.savefig(buf1, format='png', dpi=150, bbox_inches='tight')
    buf1.seek(0); plt.close(fig1)

    # ── Drawdown Curve ───────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(14, 4))
    ax2.fill_between(tn, dd_v, 0, color='#c0392b', alpha=0.55)
    ax2.plot(tn, dd_v, color='#c0392b', linewidth=1.3)
    ax2.axhline(0, color='grey', linestyle='--', linewidth=0.7)
    mi = int(np.argmin(dd_v))
    if dd_v[mi] < 0:
        ax2.annotate(
            f"Max DD: {dd_v[mi]:.2f}%",
            xy=(tn[mi], dd_v[mi]),
            xytext=(tn[mi] + max(len(tn)//15, 2), dd_v[mi] * 0.5),
            arrowprops=dict(arrowstyle='->', color='#7b241c', lw=0.9),
            fontsize=9, color='#7b241c', fontweight='bold'
        )
    ax2.set_title('BankNifty Short Strangle — Drawdown from Peak (%)',
                  fontsize=13, fontweight='bold', pad=10)
    ax2.set_xlabel('Trade Number', fontsize=10)
    ax2.set_ylabel('Drawdown %', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    plt.tight_layout()
    plt.show()
    buf2 = BytesIO()
    fig2.savefig(buf2, format='png', dpi=150, bbox_inches='tight')
    buf2.seek(0); plt.close(fig2)

    STEP_TIMES['8_charts'] = time.time() - t0
    print(f"[STEP 8]  Charts rendered   Time: {STEP_TIMES['8_charts']:.2f}s")
    return buf1, buf2


# ════════════════════════════════════════════════════════════
#  EXCEL STYLE CONSTANTS
# ════════════════════════════════════════════════════════════
_NAVY   = '1F497D'
_LTBLUE = 'DCE6F1'
_SUBHDR = 'BDD7EE'
_WHITE  = 'FFFFFF'
_GREEN  = 'C6EFCE'
_RED    = 'FFC7CE'
_YELLOW = 'FFEB9C'
_ALTROW = 'F2F7FC'

H_FILL  = PatternFill('solid', start_color=_NAVY)
A_FILL  = PatternFill('solid', start_color=_ALTROW)
W_FILL  = PatternFill('solid', start_color=_WHITE)
G_FILL  = PatternFill('solid', start_color=_GREEN)
R_FILL  = PatternFill('solid', start_color=_RED)
B_FILL  = PatternFill('solid', start_color=_SUBHDR)
Y_FILL  = PatternFill('solid', start_color=_YELLOW)

H_FONT  = Font(bold=True, color=_WHITE,  name='Calibri', size=10)
B_FONT  = Font(bold=True, color='000000',name='Calibri', size=10)
N_FONT  = Font(name='Calibri', size=9)
T_FONT  = Font(bold=True, name='Calibri', size=10, color=_NAVY)
BDR     = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'),  bottom=Side(style='thin'))
CTR     = Alignment(horizontal='center', vertical='center', wrap_text=True)
LFT     = Alignment(horizontal='left',   vertical='center', wrap_text=True)


def _hrow(ws, row: int, cols: list):
    for ci, val in enumerate(cols, 1):
        c = ws.cell(row=row, column=ci, value=val)
        c.font = H_FONT; c.fill = H_FILL; c.alignment = CTR; c.border = BDR


def _cell(ws, row, col, val, font=None, fill=None, align=None, num_fmt=None):
    c = ws.cell(row=row, column=col, value=val)
    c.font = font or N_FONT
    c.fill = fill or W_FILL
    c.alignment = align or CTR
    c.border = BDR
    if num_fmt:
        c.number_format = num_fmt
    return c


def _autowidth(ws, mn=10, mx=30):
    for col in ws.columns:
        w = max((len(str(c.value)) if c.value is not None else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(w+2, mn), mx)


# ════════════════════════════════════════════════════════════
#  STEP 9 — WRITE EXCEL WORKBOOK (4 sheets)
# ════════════════════════════════════════════════════════════
def write_excel(ts: pd.DataFrame, stats: dict,
                buf_eq: BytesIO, buf_dd: BytesIO,
                path: str, total_elapsed: float, step_times: dict):
    t0 = time.time()
    print("[STEP 9]  Writing Excel workbook ...")
    wb = Workbook()

    # ══════════════════════════════════════════════════════════
    #  SHEET 1 — GUIDE
    # ══════════════════════════════════════════════════════════
    ws_g = wb.active
    ws_g.title = 'Guide'
    ws_g.sheet_view.showGridLines = False
    ws_g.column_dimensions['A'].width = 100

    guide = [
        ("BANKNIFTY SHORT STRANGLE BACKTEST — GUIDE & DOCUMENTATION", "title"),
        ("", "blank"),
        ("OBJECTIVE", "section"),
        ("Backtest a 09:20 short strangle trading system on BankNifty using one year "
         "of 1-minute options and index data (2023). The system sells one CE and one PE "
         "option each Week-1 trading day, selecting strikes whose premium is closest "
         "to Rs.50 at 09:20, and exits at 15:20 or on a 50% stop-loss.", "body"),
        ("", "blank"),
        ("STRATEGY OVERVIEW", "section"),
        ("A SHORT STRANGLE involves simultaneously SELLING a call (CE) and a put (PE) "
         "option. The strategy profits from time decay when the underlying stays within "
         "a range. Since both options are sold, the trader RECEIVES premium upfront. "
         "Risk is if BankNifty moves sharply, causing one leg to trigger the stop-loss.", "body"),
        ("", "blank"),
        ("TASK BREAKDOWN", "section"),
        ("1. STRIKE SELECTION MODULE", "sub"),
        ("   At 09:20 each Week-1 trading day, scan all available CE and PE strikes. "
         "   Select the one strike per type whose 1-minute Close price is closest to "
         "   Rs.50. This ensures we sell near-the-money options with a defined premium.", "body"),
        ("", "blank"),
        ("2. SIGNAL GENERATION MODULE", "sub"),
        (f"   ENTRY  : {ENTRY_TIME} candle close price (first traded bar after open)", "body"),
        (f"   EXIT   : {EXIT_TIME} candle close price, OR stop-loss — whichever comes first", "body"),
        (f"   SL     : 50% of entry price. Monitor the 'High' column of each 1-min bar.", "body"),
        ("   If any bar's High >= Entry Price x 1.50, exit at that SL level immediately.", "body"),
        ("   (We SOLD options first — rising price = loss for us)", "body"),
        ("", "blank"),
        ("3. POSITION SIZING MODULE", "sub"),
        (f"   Fixed 1 lot per leg per day. Lot size = {LOT_SIZE}. Quantity = {QUANTITY} units.", "body"),
        ("   No compounding — the same 1 lot is traded regardless of running P&L.", "body"),
        ("", "blank"),
        ("4. UNIVERSE / SCHEDULE", "sub"),
        ("   Trade ONLY Week-1 days: from Monday to the first Wednesday of each month.", "body"),
        ("   The first Wednesday is the EXPIRY day for that month's weekly options.", "body"),
        ("   A trade is taken EVERY Week-1 day (both expiry and non-expiry).", "body"),
        ("", "blank"),
        ("OUTPUT — SHEETS IN THIS WORKBOOK", "section"),
        ("Tradesheet  ->  One row per option leg (CE and PE) per day. Contains all "
         "required trade fields: dates, times, ticker, strike, type, prices, "
         "quantity, values, P&L, cumulative P&L, capital, spot price, and NAV.", "body"),
        ("Statistics  ->  All required statistics: CAGR, Max Drawdown, Winner/Loser "
         "analysis, Average % P&L (expiry vs non-expiry), Monthly P&L table, "
         "and code timing breakdown.", "body"),
        ("Charts      ->  Equity curve (trade-wise NAV) and drawdown chart, both "
         "embedded as high-resolution images.", "body"),
        ("", "blank"),
        ("KEY FORMULAS", "section"),
        ("Entry/Exit Value    = Price x Quantity", "body"),
        ("Gross P&L per leg   = (Entry Price - Exit Price) x Quantity", "body"),
        ("  [Positive = profit (option price fell), Negative = loss (option price rose)]", "body"),
        ("Cumulative P&L      = Running sum of Gross P&L across all trades (trade-wise)", "body"),
        ("Available Capital   = Starting Capital + Cumulative Daily P&L (day-level)", "body"),
        ("NAV (Equity Curve)  = 100 + (Cumulative Trade P&L / Starting Capital) x 100", "body"),
        ("CAGR                = (Final NAV / Starting NAV) ^ (1 / Years) - 1", "body"),
        ("Max Drawdown        = Max percentage decline from any running NAV peak to trough", "body"),
        ("Monthly P&L %       = (End NAV this month / End NAV previous month - 1) x 100", "body"),
        ("Win Rate            = Trades with Gross P&L > 0 / Total Trades x 100", "body"),
        ("", "blank"),
        ("CONFIGURATION USED", "section"),
        (f"Starting Capital   : Rs. {INITIAL_CAPITAL:,.0f}", "body"),
        (f"Base NAV           : {STARTING_NAV}", "body"),
        (f"Lot Size           : {LOT_SIZE}  |  Lots per day: {NUM_LOTS}  |  Quantity: {QUANTITY}", "body"),
        (f"Target Premium     : Rs. {TARGET_PREM}", "body"),
        (f"Stop-Loss          : {int(SL_PCT*100)}% of entry price (High-based detection)", "body"),
        (f"Entry Time         : {ENTRY_TIME} candle close", "body"),
        (f"Exit Time          : {EXIT_TIME} candle close (if no SL)", "body"),
        ("", "blank"),
        ("HOW TO INTERPRET THE TRADESHEET", "section"),
        ("- Each row = one option leg. Two rows per day (one CE, one PE).", "body"),
        ("- SL Hit = YES means the stop-loss triggered before 15:20.", "body"),
        ("- Exit Time shows the actual time of exit (15:20 or the SL bar's time).", "body"),
        ("- Gross P&L is green (profit) or red (loss) in the sheet.", "body"),
        ("- Cumulative P&L is trade-wise (not daily), matching the NAV curve.", "body"),
        ("- Available Capital resets to end-of-day level for both legs of the same day.", "body"),
        ("- Is Expiry = YES means that day is the first Wednesday (weekly expiry).", "body"),
        ("", "blank"),
        ("CODE TIMING (seconds)", "section"),
    ]
    for k, v in step_times.items():
        guide.append((f"   {k:<42} {v:.2f} s", "body"))
    guide.append(("", "blank"))
    guide.append((f"   {'TOTAL RUN TIME':<42} {total_elapsed:.1f} s", "sub"))
    guide.append(("", "blank"))
    guide.append(("   Target: < 60 seconds end-to-end (including all data processing)", "body"))

    r = 1
    for text, kind in guide:
        c = ws_g.cell(row=r, column=1, value=text)
        if   kind == "title":
            c.font = Font(bold=True, size=14, name='Calibri', color=_NAVY)
            ws_g.row_dimensions[r].height = 22
        elif kind == "section":
            c.font = Font(bold=True, size=11, name='Calibri', color=_NAVY)
            ws_g.row_dimensions[r].height = 18
        elif kind == "sub":
            c.font = Font(bold=True, size=10, name='Calibri', color='000000')
        else:
            c.font = Font(size=10, name='Calibri', color='000000')
        c.alignment = Alignment(horizontal='left', vertical='top', wrap_text=True)
        if kind not in ('blank',):
            ws_g.row_dimensions[r].height = max(ws_g.row_dimensions[r].height or 15, 15)
        r += 1

    # ══════════════════════════════════════════════════════════
    #  SHEET 2 — TRADESHEET
    # ══════════════════════════════════════════════════════════
    ws_t = wb.create_sheet('Tradesheet')
    ws_t.sheet_view.showGridLines = False
    ws_t.row_dimensions[1].height = 28

    ts_headers = [
        'Entry Date', 'Entry Time', 'Exit Date', 'Exit Time',
        'Option Ticker', 'Strike Price', 'Option Type',
        'Entry Price', 'Exit Price', 'SL Hit',
        'Quantity', 'Entry Value', 'Exit Value',
        'Gross P&L', 'Cumulative P&L', 'P&L %',
        'Available Capital', 'BankNifty Spot Close', 'Is Expiry', 'NAV'
    ]
    _hrow(ws_t, 1, ts_headers)

    for ri, (_, row) in enumerate(ts.iterrows(), 2):
        pnl = float(row['GrossPnL'])
        bg  = A_FILL if ri % 2 == 0 else W_FILL

        spot_val = row.get('Spot_Close')
        spot_out = round(float(spot_val), 2) if pd.notna(spot_val) and spot_val else ''

        row_vals = [
            str(row['Date']),
            ENTRY_TIME,
            str(row['Date']),
            str(row.get('ExitTime', EXIT_TIME)),
            str(row['EntryTicker']),
            int(row['Strike']),
            str(row['OptionType']),
            round(float(row['EntryPrice']), 2),
            round(float(row['ExitPrice']),  2),
            'YES' if bool(row.get('SL_Hit', False)) else 'NO',
            int(row['Quantity']),
            round(float(row['EntryValue']), 2),
            round(float(row['ExitValue']),  2),
            round(pnl, 2),
            round(float(row['CumPnL']), 2),
            round(float(row['PnL_Pct']), 4),
            round(float(row['AvailCap']), 2),
            spot_out,
            'YES' if bool(row['IsExpiry']) else 'NO',
            round(float(row['NAV']), 4),
        ]
        for ci, val in enumerate(row_vals, 1):
            # Colour Gross P&L column
            if ci == 14:
                fill = G_FILL if pnl > 0 else (R_FILL if pnl < 0 else bg)
            # Colour SL Hit column
            elif ci == 10:
                fill = (PatternFill('solid', start_color='FFE0CC')
                        if val == 'YES' else bg)
            else:
                fill = bg
            _cell(ws_t, ri, ci, val, fill=fill)

    ws_t.freeze_panes = 'A2'
    _autowidth(ws_t, mn=11, mx=28)

    # ══════════════════════════════════════════════════════════
    #  SHEET 3 — STATISTICS
    # ══════════════════════════════════════════════════════════
    ws_s = wb.create_sheet('Statistics')
    ws_s.sheet_view.showGridLines = False
    ws_s.column_dimensions['A'].width = 32
    ws_s.column_dimensions['B'].width = 18
    ws_s.column_dimensions['C'].width = 18
    ws_s.column_dimensions['D'].width = 18

    def s_title(row, text):
        ws_s.merge_cells(start_row=row, start_column=1, end_row=row, end_column=4)
        c = ws_s.cell(row=row, column=1, value=text)
        c.font      = Font(bold=True, color=_WHITE, name='Calibri', size=10)
        c.fill      = PatternFill('solid', start_color=_NAVY)
        c.alignment = LFT
        c.border    = BDR
        ws_s.row_dimensions[row].height = 20

    def s_sub(row, cols):
        for ci, v in enumerate(cols, 1):
            c = ws_s.cell(row=row, column=ci, value=v)
            c.font = B_FONT; c.fill = B_FILL; c.alignment = CTR; c.border = BDR

    def s_row(row, cols, hl=None):
        for ci, v in enumerate(cols, 1):
            fill = Y_FILL if ci == hl else (A_FILL if row % 2 == 0 else W_FILL)
            _cell(ws_s, row, ci, v, fill=fill)

    def s_blank(row):
        ws_s.row_dimensions[row].height = 8

    r = 1

    # ── Performance Summary ───────────────────────────────────
    s_title(r, "1.  PERFORMANCE SUMMARY"); r += 1
    s_sub(r, ["Metric", "Value", "", ""]); r += 1
    perf = [
        ("CAGR (%)",                      stats['CAGR_%']),
        ("Max Drawdown (%)",               stats['Max_Drawdown_%']),
        ("Total Week-1 Trading Days",      ts['Date'].nunique()),
        ("Total Option Legs Traded",       len(ts)),
        ("Starting Capital (Rs.)",         f"{INITIAL_CAPITAL:,}"),
        ("Starting NAV",                   STARTING_NAV),
        ("Ending NAV",                     round(float(ts['NAV'].iloc[-1]), 4)),
        ("Total Gross P&L (Rs.)",          round(float(ts['GrossPnL'].sum()), 2)),
    ]
    for label, val in perf:
        s_row(r, [label, val, "", ""], hl=2); r += 1
    r += 1; s_blank(r); r += 1

    # ── Winner / Loser Analysis ───────────────────────────────
    s_title(r, "2.  WINNER / LOSER ANALYSIS"); r += 1
    s_sub(r, ["Metric", "CE (Calls)", "PE (Puts)", "Combined"]); r += 1
    wl_metrics = [
        ("Number of Winners", 'CE_Winners',  'PE_Winners',  'Both_Winners'),
        ("Number of Losers",  'CE_Losers',   'PE_Losers',   'Both_Losers'),
        ("Total Trades",      'CE_Total',    'PE_Total',    'Both_Total'),
        ("Win %",             'CE_Win_Pct',  'PE_Win_Pct',  'Both_Win_Pct'),
        ("Loss %",            'CE_Loss_Pct', 'PE_Loss_Pct', 'Both_Loss_Pct'),
    ]
    for label, ck, pk, bk in wl_metrics:
        s_row(r, [label, stats[ck], stats[pk], stats[bk]]); r += 1
    r += 1; s_blank(r); r += 1

    # ── Average % P&L ─────────────────────────────────────────
    s_title(r, "3.  AVERAGE % P&L  (Expiry Days vs Non-Expiry Days)"); r += 1
    s_sub(r, ["Day Type", "CE Avg P&L %", "PE Avg P&L %", "Combined Avg P&L %"]); r += 1
    for label, suffix in [("Expiry Days (Wednesday)", "Expiry"),
                           ("Non-Expiry Days (Mon/Tue)", "NonExpiry")]:
        vals = [label,
                stats.get(f'CE_{suffix}_Avg_PnL_%', ''),
                stats.get(f'PE_{suffix}_Avg_PnL_%', ''),
                stats.get(f'Both_{suffix}_Avg_PnL_%', '')]
        s_row(r, vals); r += 1
    r += 1; s_blank(r); r += 1

    # ── Monthly P&L Table ─────────────────────────────────────
    s_title(r, "4.  MONTHLY P&L TABLE  (NAV-Based)"); r += 1
    s_sub(r, ["Month", "End NAV", "Prev Month End NAV", "Monthly P&L %"]); r += 1
    mdf = stats['Monthly_PnL_Table'].reset_index()
    for _, mr in mdf.iterrows():
        pct   = float(mr['Monthly_PnL%'])
        pfill = G_FILL if pct > 0 else (R_FILL if pct < 0 else W_FILL)
        row_vals = [
            str(mr['YM']),
            round(float(mr['EndNAV']),  4),
            round(float(mr['PrevNAV']), 4),
            round(pct, 2),
        ]
        for ci, val in enumerate(row_vals, 1):
            c = _cell(ws_s, r, ci, val,
                      fill=(A_FILL if r % 2 == 0 else W_FILL))
            if ci == 4:
                c.fill = pfill
        r += 1
    r += 1; s_blank(r); r += 1

    # ── Code Timing ───────────────────────────────────────────
    s_title(r, "5.  CODE EXECUTION TIMING (seconds)"); r += 1
    s_sub(r, ["Step", "Time (s)", "", ""]); r += 1
    for sname, sec in step_times.items():
        s_row(r, [sname, round(sec, 2), "", ""]); r += 1
    s_row(r, ["TOTAL RUN TIME", round(total_elapsed, 2), "", ""], hl=2); r += 1
    s_row(r, ["Target", "< 60 seconds", "", ""]); r += 1

    # ══════════════════════════════════════════════════════════
    #  SHEET 4 — CHARTS
    # ══════════════════════════════════════════════════════════
    ws_c = wb.create_sheet('Charts')
    ws_c.sheet_view.showGridLines = False

    hdr = ws_c.cell(row=1, column=1,
                    value='BankNifty Short Strangle — Equity Curve & Drawdown Chart')
    hdr.font = Font(bold=True, size=13, name='Calibri', color=_NAVY)
    ws_c.row_dimensions[1].height = 22

    sub1 = ws_c.cell(row=2, column=1, value='Equity Curve (Base NAV = 100) — Trade-wise')
    sub1.font = Font(bold=True, size=10, name='Calibri', color='444444')

    img1 = XLImage(buf_eq); img1.width = 960; img1.height = 350
    ws_c.add_image(img1, 'A3')

    sub2 = ws_c.cell(row=26, column=1, value='Drawdown from Peak (%)')
    sub2.font = Font(bold=True, size=10, name='Calibri', color='444444')

    img2 = XLImage(buf_dd); img2.width = 960; img2.height = 290
    ws_c.add_image(img2, 'A27')

    wb.save(path)
    elapsed = time.time() - t0
    STEP_TIMES['9_write_excel'] = elapsed
    print(f"          Saved -> {path}")
    print(f"          Time: {elapsed:.1f}s")


# ════════════════════════════════════════════════════════════
#  MAIN
# ════════════════════════════════════════════════════════════
def main():
    GRAND_START = time.time()

    print("=" * 62)
    print("  BANKNIFTY SHORT STRANGLE BACKTEST — FINAL SUBMISSION")
    print("=" * 62)

    df          = load_options(OPTIONS_PATH)
    spot        = load_spot(SPOT_PATH)
    week1_dates = get_week1_dates(df['Date'].unique())
    selected    = select_strikes(df, week1_dates)
    merged      = get_exits(df, selected)
    ts          = build_tradesheet(merged, spot, week1_dates)
    stats       = compute_stats(ts)
    buf_eq, buf_dd = make_charts(stats)

    grand_total = time.time() - GRAND_START
    write_excel(ts, stats, buf_eq, buf_dd,
                OUTPUT_PATH, grand_total, STEP_TIMES)

    # Auto-download in Google Colab
    try:
        from google.colab import files
        files.download(OUTPUT_PATH)
        print("\n  Excel file downloading to your computer ...")
    except ImportError:
        print(f"\n  File saved locally: {OUTPUT_PATH}")

    # Final console summary
    print("\n" + "=" * 62)
    print("  BACKTEST RESULTS SUMMARY")
    print("=" * 62)
    print(f"  Total Trades (legs)   : {len(ts)}")
    print(f"  Week-1 Trading Days   : {ts['Date'].nunique()}")
    print(f"  Starting Capital      : Rs. {INITIAL_CAPITAL:,}")
    print(f"  Total Gross P&L       : Rs. {ts['GrossPnL'].sum():,.2f}")
    print(f"  CAGR                  : {stats['CAGR_%']:.2f}%")
    print(f"  Max Drawdown          : {stats['Max_Drawdown_%']:.2f}%")
    print(f"  CE  Win Rate          : {stats['CE_Win_Pct']:.1f}%  "
          f"({stats['CE_Winners']}W / {stats['CE_Losers']}L)")
    print(f"  PE  Win Rate          : {stats['PE_Win_Pct']:.1f}%  "
          f"({stats['PE_Winners']}W / {stats['PE_Losers']}L)")
    print(f"  Combined Win Rate     : {stats['Both_Win_Pct']:.1f}%")
    print(f"  Starting NAV          : {STARTING_NAV}")
    print(f"  Ending NAV            : {round(float(ts['NAV'].iloc[-1]), 4)}")
    print(f"  Total Run Time        : {grand_total:.1f}s")
    print("=" * 62)

    print("\n  STEP-BY-STEP TIMING")
    print("  " + "-" * 46)
    for step, sec in STEP_TIMES.items():
        bar   = '#' * int(sec / grand_total * 30)
        print(f"  {step:<36} {sec:>6.2f}s  {bar}")
    print(f"  {'TOTAL':<36} {grand_total:>6.1f}s")
    print("=" * 62)


if __name__ == '__main__':
    main()

  BANKNIFTY SHORT STRANGLE BACKTEST — FINAL SUBMISSION
[STEP 1]  Loading options data ...
          Rows: 10,266,681   Columns: ['Unnamed: 0', 'Date', 'Ticker', 'Time', 'Open', 'High', 'Low', 'Close', 'OptionType', 'Strike']
          Time: 36.4s
[STEP 2]  Loading BankNifty spot data ...
          Rows: 136,324   Dates: 364
          Time: 1.0s
[STEP 3]  Identifying Week-1 trading dates ...
          Week-1 trading days : 26
          Expiry days (Wed)   : 12
          Time: 0.12s
[STEP 4]  Strike selection (closest to Rs.50 at 09:20) ...
          Legs selected : 52  (26 days x 2 legs)
          Time: 6.83s
[STEP 5]  Computing exits and stop-losses (vectorised) ...
          SL triggered : 23 legs
          Time: 2.72s
[STEP 6]  Building trade sheet ...
          Rows built : 52
          Time: 0.12s
[STEP 7]  Computing statistics ...
          CAGR             : 3.70%
          Max Drawdown     : -0.22%
          Combined Win Rate: 55.8%
          Time: 0.01s
[STEP 8]  Charts rendere

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  Excel file downloading to your computer ...

  BACKTEST RESULTS SUMMARY
  Total Trades (legs)   : 52
  Week-1 Trading Days   : 26
  Starting Capital      : Rs. 1,000,000
  Total Gross P&L       : Rs. 3,757.95
  CAGR                  : 3.70%
  Max Drawdown          : -0.22%
  CE  Win Rate          : 57.7%  (15W / 11L)
  PE  Win Rate          : 53.9%  (14W / 12L)
  Combined Win Rate     : 55.8%
  Starting NAV          : 100.0
  Ending NAV            : 100.3758
  Total Run Time        : 48.4s

  STEP-BY-STEP TIMING
  ----------------------------------------------
  1_load_options                        36.39s  ######################
  2_load_spot                            1.01s  
  3_week1_dates                          0.12s  
  4_strike_selection                     6.83s  ####
  5_exit_logic                           2.72s  #
  6_tradesheet                           0.12s  
  7_statistics                           0.01s  
  8_charts                               0.61s  
  9_write_e

In [ ]:
"""
==============================================================
  BankNifty Short Strangle Backtest — FINAL SUBMISSION
==============================================================
Strategy  : Short Strangle — Sell CE + Sell PE simultaneously
Universe  : Week-1 trading days only (Mon → first Wednesday of each month)
Entry     : 09:20 candle close, strike with premium closest to Rs.50
Exit      : 15:20 candle close OR 50% stop-loss (whichever comes first)
SL check  : Uses High column of every 1-minute bar after entry
Quantity  : 1 lot = 15 units per leg, per day. No compounding.
Expiry    : First Wednesday of each month = weekly expiry day

Optimisations
  • usecols + plain-str dtype on read_csv  → fast CSV load (~40s)
  • Vectorised strike selection             → idxmin groupby, no loop
  • Vectorised exit / SL logic             → merge-based, no loop
  Total run time : ~53 seconds (under 60s target)

Bug-fixes (vs original submission)
  1. SL exit price: SL_Level never propagated through intra bars
     (category/str mismatch caused wrong prices on 7 legs previously)
     Fix: SL exit = EntryPrice × 1.5 applied after final merge
  2. Spot Close: Time column zero-padded at load time so '9:20' and
     '09:20:59' both match the '09:20' startswith filter
==============================================================
"""

import time, warnings
import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot   as plt
import matplotlib.ticker   as mticker
from io              import BytesIO
from openpyxl        import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils  import get_column_letter
from openpyxl.drawing.image import Image as XLImage

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════
#  CONFIGURATION  ← set these two paths before running
# ════════════════════════════════════════════════════════════
OPTIONS_PATH    = '/path/to/options_data.csv'      # CSV or .parquet
SPOT_PATH       = '/path/to/banknifty_spot.csv'
OUTPUT_PATH     = 'BankNifty_ShortStrangle_Backtest.xlsx'

INITIAL_CAPITAL = 1_000_000          # Rs. 10,00,000
STARTING_NAV    = 100.0
LOT_SIZE        = 15
NUM_LOTS        = 1
QUANTITY        = LOT_SIZE * NUM_LOTS   # 15 per leg

TARGET_PREM     = 50.0               # select strike closest to Rs.50
SL_PCT          = 0.50               # 50% stop-loss on entry price
ENTRY_TIME      = '09:20'
EXIT_TIME       = '15:20'

STEP_TIMES: dict = {}


# ════════════════════════════════════════════════════════════
#  STEP 1 — LOAD OPTIONS DATA
# ════════════════════════════════════════════════════════════
def load_options(path: str) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 1]  Loading options data ...")

    if path.endswith('.parquet'):
        df = pd.read_parquet(path)
        df.columns = [c.strip() for c in df.columns]
    else:
        df = pd.read_csv(
            path,
            usecols=['Date', 'Ticker', 'Time', 'High', 'Close', 'Call/Put'],
            dtype={'Time': str},
            low_memory=False,
            encoding='latin1',
        )
        df.columns = [c.strip() for c in df.columns]

    if 'Call/Put' in df.columns:
        df.rename(columns={'Call/Put': 'OptionType'}, inplace=True)

    # Plain str for Ticker — avoids category/str merge bugs
    df['Date']       = pd.to_datetime(df['Date']).dt.date
    df['Time']       = df['Time'].astype(str).str[:5]
    df['Ticker']     = df['Ticker'].astype(str)

    ticker_str   = df['Ticker']
    df['Strike'] = pd.to_numeric(
        ticker_str.str.extract(r'BANKNIFTY(\d+)(?:CE|PE)', expand=False),
        errors='coerce')

    if 'OptionType' not in df.columns or df['OptionType'].isna().all():
        df['OptionType'] = ticker_str.str.extract(r'BANKNIFTY\d+(CE|PE)', expand=False)
    df['OptionType'] = df['OptionType'].astype(str).str.strip().str.upper()

    elapsed = time.time() - t0
    STEP_TIMES['1_load_options'] = elapsed
    print(f"          Rows: {len(df):,}   Columns: {list(df.columns)}")
    print(f"          Time: {elapsed:.1f}s")
    return df


# ════════════════════════════════════════════════════════════
#  STEP 2 — LOAD SPOT DATA
#  Fix: zero-pad Time so '9:20' and '09:20:59' both work
# ════════════════════════════════════════════════════════════
def load_spot(path: str) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 2]  Loading BankNifty spot data ...")

    spot = pd.read_csv(path, low_memory=False)
    spot.columns = [c.strip().lower() for c in spot.columns]

    # Auto-detect datetime and close columns
    if 'ts' in spot.columns:
        spot.rename(columns={'ts': '_dt', 'c': '_close'}, inplace=True)
    else:
        dt_col    = next((c for c in spot.columns
                          if any(k in c for k in ('date', 'time', 'dt', 'ts'))),
                         spot.columns[0])
        close_col = next((c for c in spot.columns
                          if c in ('close', 'c', 'ltp', 'last')),
                         spot.columns[-1])
        spot.rename(columns={dt_col: '_dt', close_col: '_close'}, inplace=True)

    spot['_dt']       = pd.to_datetime(spot['_dt'])
    spot['Date']      = spot['_dt'].dt.date
    # Zero-pad to HH:MM so '9:20', '09:20', '09:20:59' all normalise to '09:20'
    spot['Time']      = spot['_dt'].dt.strftime('%H:%M')
    spot['Spot_Close'] = pd.to_numeric(spot['_close'], errors='coerce')
    spot = spot[['Date', 'Time', 'Spot_Close']].copy()

    elapsed = time.time() - t0
    STEP_TIMES['2_load_spot'] = elapsed
    print(f"          Rows: {len(spot):,}   Dates: {spot['Date'].nunique()}")
    print(f"          Time: {elapsed:.1f}s")
    return spot


# ════════════════════════════════════════════════════════════
#  STEP 3 — IDENTIFY WEEK-1 TRADING DATES
#  Week-1 = Mon through first Wednesday of each calendar month
#  First Wednesday = weekly expiry day
# ════════════════════════════════════════════════════════════
def get_week1_dates(all_dates) -> list:
    t0 = time.time()
    print("[STEP 3]  Identifying Week-1 trading dates ...")

    ds    = pd.to_datetime(pd.Series(sorted(set(all_dates))))
    week1 = []

    for (y, m), grp in ds.groupby([ds.dt.year, ds.dt.month]):
        weds = grp[grp.dt.dayofweek == 2]
        if weds.empty:
            continue
        first_wed = weds.iloc[0]
        iso_week  = first_wed.isocalendar()[1]
        iso_year  = first_wed.isocalendar()[0]
        same_wk   = grp[
            (grp.dt.isocalendar().week.values == iso_week) &
            (grp.dt.isocalendar().year.values == iso_year) &
            (grp <= first_wed)
        ]
        week1.extend(same_wk.dt.date.tolist())

    week1  = sorted(set(week1))
    expiry = [d for d in week1 if pd.Timestamp(d).dayofweek == 2]
    elapsed = time.time() - t0
    STEP_TIMES['3_week1_dates'] = elapsed
    print(f"          Week-1 trading days : {len(week1)}")
    print(f"          Expiry days (Wed)   : {len(expiry)}")
    print(f"          Time: {elapsed:.2f}s")
    return week1


# ════════════════════════════════════════════════════════════
#  STEP 4 — STRIKE SELECTION MODULE (vectorised)
#  At 09:20 on each Week-1 day, pick the CE and PE strikes
#  whose 1-min Close is closest to Rs.50 (target premium)
# ════════════════════════════════════════════════════════════
def select_strikes(df: pd.DataFrame, week1_dates: list) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 4]  Strike selection (closest to Rs.50 at 09:20) ...")

    mask = (
        df['Time'].str.startswith(ENTRY_TIME) &
        df['Date'].isin(set(week1_dates)) &
        df['OptionType'].isin(['CE', 'PE'])
    )
    entry = df.loc[mask, ['Date', 'Ticker', 'Strike', 'OptionType', 'Close']].copy()
    entry['Dist'] = (entry['Close'] - TARGET_PREM).abs()

    # Pure vectorised — groupby idxmin, no Python loop
    idx      = entry.groupby(['Date', 'OptionType'], sort=False)['Dist'].idxmin()
    selected = entry.loc[idx].reset_index(drop=True)
    selected.rename(columns={'Close': 'EntryPrice', 'Ticker': 'EntryTicker'}, inplace=True)

    # SL_Level stored only here — never propagated into intra bars
    selected['SL_Level'] = (selected['EntryPrice'] * (1.0 + SL_PCT)).round(2)

    elapsed = time.time() - t0
    STEP_TIMES['4_strike_selection'] = elapsed
    print(f"          Legs selected : {len(selected)}  ({len(selected)//2} days x 2 legs)")
    print(f"          Time: {elapsed:.2f}s")
    return selected


# ════════════════════════════════════════════════════════════
#  STEP 5 — SIGNAL GENERATION + EXIT LOGIC (fully vectorised)
#
#  Entry  : 09:20 close
#  Exit   : 15:20 close  OR  first bar where High >= SL_Level
#  SL price = EntryPrice × 1.5 (set from selected, not from intra)
#
#  Key: SL_Level only used for trigger detection inside intra bars.
#  Actual SL exit price is assigned AFTER merge from selected.SL_Level
#  This avoids any dtype mismatch corrupting the exit price.
# ════════════════════════════════════════════════════════════
def get_exits(df: pd.DataFrame, selected: pd.DataFrame) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 5]  Computing exits and stop-losses (vectorised) ...")

    traded_tickers = set(selected['EntryTicker'])
    traded_dates   = set(selected['Date'])

    # Intraday bars after entry for traded legs only
    intra = df[
        (df['Time'] > ENTRY_TIME) &
        df['Date'].isin(traded_dates) &
        df['Ticker'].isin(traded_tickers)
    ][['Date', 'Ticker', 'Time', 'High', 'Close']].copy()

    # Attach SL_Level for trigger detection only
    sl_ref = selected[['EntryTicker', 'SL_Level']].rename(columns={'EntryTicker': 'Ticker'})
    intra  = intra.merge(sl_ref, on='Ticker', how='left')

    # 1. First bar where High >= SL_Level
    sl_hit   = intra[intra['High'] >= intra['SL_Level']]
    sl_first = (
        sl_hit.sort_values('Time')
        .groupby(['Date', 'Ticker'], sort=False)
        .agg(SL_Time=('Time', 'first'))
        .reset_index()
    )

    # 2. Normal exit at 15:20 close
    norm_exit = (
        intra[intra['Time'].str.startswith(EXIT_TIME)]
        .groupby(['Date', 'Ticker'], sort=False)
        .agg(Norm_Time=('Time', 'last'), Norm_Close=('Close', 'last'))
        .reset_index()
    )

    # 3. Fallback: last available bar of the day
    last_bar = (
        intra.sort_values('Time')
        .groupby(['Date', 'Ticker'], sort=False)
        .agg(Last_Time=('Time', 'last'), Last_Close=('Close', 'last'))
        .reset_index()
    )

    # 4. Merge onto selected
    res = selected.copy()
    res = res.merge(sl_first,  left_on=['Date','EntryTicker'],
                    right_on=['Date','Ticker'], how='left').drop(columns='Ticker')
    res = res.merge(norm_exit, left_on=['Date','EntryTicker'],
                    right_on=['Date','Ticker'], how='left').drop(columns='Ticker')
    res = res.merge(last_bar,  left_on=['Date','EntryTicker'],
                    right_on=['Date','Ticker'], how='left').drop(columns='Ticker')

    # 5. Resolve exit — SL price from selected.SL_Level (always correct)
    res['SL_Hit'] = res['SL_Time'].notna()

    res['ExitPrice'] = np.where(
        res['SL_Hit'],
        res['SL_Level'],                                    # EntryPrice * 1.5
        np.where(
            res['Norm_Close'].notna(),
            res['Norm_Close'],                              # 15:20 close
            res['Last_Close'].fillna(res['EntryPrice'])     # fallback
        )
    ).astype(float).round(2)

    res['ExitTime'] = np.where(
        res['SL_Hit'],
        res['SL_Time'],
        np.where(res['Norm_Time'].notna(), res['Norm_Time'],
                 res['Last_Time'].fillna(EXIT_TIME))
    )

    res.drop(columns=['SL_Level','SL_Time','Norm_Time','Norm_Close',
                       'Last_Time','Last_Close'], inplace=True)

    elapsed = time.time() - t0
    STEP_TIMES['5_exit_logic'] = elapsed
    print(f"          SL triggered : {res['SL_Hit'].sum()} legs")
    print(f"          Time: {elapsed:.2f}s")
    return res


# ════════════════════════════════════════════════════════════
#  STEP 6 — POSITION SIZING + BUILD TRADE SHEET
# ════════════════════════════════════════════════════════════
def build_tradesheet(merged: pd.DataFrame, spot: pd.DataFrame,
                     week1_dates: list) -> pd.DataFrame:
    t0 = time.time()
    print("[STEP 6]  Building trade sheet ...")

    merged['Date'] = pd.to_datetime(merged['Date']).dt.date
    spot['Date']   = pd.to_datetime(spot['Date']).dt.date

    # Spot close at 09:20 entry time — one row per day
    spot_entry = (
        spot[spot['Time'].str.startswith(ENTRY_TIME)]
        [['Date', 'Spot_Close']]
        .drop_duplicates('Date')
    )
    merged = merged.merge(spot_entry, on='Date', how='left')

    # Expiry flag
    expiry_dates       = {d for d in week1_dates if pd.Timestamp(d).dayofweek == 2}
    merged['IsExpiry'] = merged['Date'].isin(expiry_dates)

    # Position sizing — fixed 1 lot per leg, no compounding
    merged['Quantity']   = QUANTITY
    merged['EntryValue'] = (merged['EntryPrice'] * QUANTITY).round(2)
    merged['ExitValue']  = (merged['ExitPrice']  * QUANTITY).round(2)

    # P&L — short position: profit when price falls
    merged['GrossPnL'] = ((merged['EntryPrice'] - merged['ExitPrice']) * QUANTITY).round(2)
    merged['PnL_Pct']  = ((merged['EntryPrice'] - merged['ExitPrice'])
                           / merged['EntryPrice'] * 100).round(4)

    # Sort chronologically, CE before PE each day
    merged = (merged
              .sort_values(['Date', 'OptionType'], ascending=[True, False])
              .reset_index(drop=True))

    # Available capital — day-level (cumulative daily P&L)
    day_pnl             = (merged.groupby('Date')['GrossPnL']
                           .sum().rename('DailyPnL').reset_index())
    day_pnl['DayCumPnL'] = day_pnl['DailyPnL'].cumsum()
    day_pnl['AvailCap']  = (INITIAL_CAPITAL + day_pnl['DayCumPnL']).round(2)
    merged = merged.merge(day_pnl[['Date','AvailCap']], on='Date', how='left')

    # Trade-wise cumulative P&L and NAV (equity curve)
    merged['CumPnL'] = merged['GrossPnL'].cumsum().round(2)
    merged['NAV']    = (STARTING_NAV
                        + merged['CumPnL'] / INITIAL_CAPITAL * STARTING_NAV
                        ).round(4)

    elapsed = time.time() - t0
    STEP_TIMES['6_tradesheet'] = elapsed
    print(f"          Rows built : {len(merged)}")
    print(f"          Time: {elapsed:.2f}s")
    return merged


# ════════════════════════════════════════════════════════════
#  STEP 7 — STATISTICAL ANALYSIS
# ════════════════════════════════════════════════════════════
def compute_stats(ts: pd.DataFrame) -> dict:
    t0 = time.time()
    print("[STEP 7]  Computing statistics ...")

    nav    = ts['NAV'].values
    n_days = ts['Date'].nunique()
    years  = n_days / 252

    # CAGR
    cagr = ((nav[-1] / STARTING_NAV) ** (1 / max(years, 1/252)) - 1) * 100

    # Max Drawdown — trade-wise from running NAV peak
    run_peak = np.maximum.accumulate(nav)
    dd_arr   = (nav - run_peak) / run_peak * 100
    max_dd   = dd_arr.min()

    stats = {
        'CAGR_%'        : round(cagr,   2),
        'Max_Drawdown_%': round(max_dd, 2),
    }

    # Winner / loser counts
    for opt in ['CE', 'PE', 'Both']:
        sub = ts if opt == 'Both' else ts[ts['OptionType'] == opt]
        w = (sub['GrossPnL'] > 0).sum()
        l = (sub['GrossPnL'] <= 0).sum()
        t = len(sub)
        stats.update({
            f'{opt}_Winners' : int(w),
            f'{opt}_Losers'  : int(l),
            f'{opt}_Total'   : int(t),
            f'{opt}_Win_Pct' : round(w/t*100, 2) if t else 0,
            f'{opt}_Loss_Pct': round(l/t*100, 2) if t else 0,
        })

    # Average % P&L — expiry vs non-expiry
    for opt in ['CE', 'PE', 'Both']:
        sub = ts if opt == 'Both' else ts[ts['OptionType'] == opt]
        for flag, lbl in [(True, 'Expiry'), (False, 'NonExpiry')]:
            g = sub[sub['IsExpiry'] == flag]['PnL_Pct']
            stats[f'{opt}_{lbl}_Avg_PnL_%'] = round(g.mean(), 4) if len(g) else float('nan')

    # Monthly P&L table
    tc         = ts.copy()
    tc['Date'] = pd.to_datetime(tc['Date'])
    tc['YM']   = tc['Date'].dt.to_period('M')
    mnav       = tc.groupby('YM')['NAV'].last()
    mp         = pd.DataFrame({'EndNAV': mnav})
    mp['PrevNAV']      = mp['EndNAV'].shift(1).fillna(STARTING_NAV)
    mp['Monthly_PnL%'] = ((mp['EndNAV'] - mp['PrevNAV']) / mp['PrevNAV'] * 100).round(2)

    stats['Monthly_PnL_Table'] = mp
    stats['NAV_Series']        = pd.Series(nav)
    stats['DD_Series']         = pd.Series(dd_arr)

    elapsed = time.time() - t0
    STEP_TIMES['7_statistics'] = elapsed
    print(f"          CAGR             : {cagr:.2f}%")
    print(f"          Max Drawdown     : {max_dd:.2f}%")
    print(f"          Combined Win Rate: {stats['Both_Win_Pct']:.1f}%")
    print(f"          Time: {elapsed:.2f}s")
    return stats


# ════════════════════════════════════════════════════════════
#  STEP 8 — CHARTS (equity curve + drawdown)
# ════════════════════════════════════════════════════════════
def make_charts(stats: dict) -> tuple:
    t0    = time.time()
    nav_v = stats['NAV_Series'].values
    dd_v  = stats['DD_Series'].values
    tn    = np.arange(1, len(nav_v) + 1)

    # ── Equity Curve ─────────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(14, 5))
    ax1.plot(tn, nav_v, color='#1a6496', linewidth=1.8, zorder=3, label='NAV')
    ax1.fill_between(tn, STARTING_NAV, nav_v,
                     where=(nav_v >= STARTING_NAV),
                     alpha=0.18, color='#27ae60', label='Above base NAV')
    ax1.fill_between(tn, STARTING_NAV, nav_v,
                     where=(nav_v < STARTING_NAV),
                     alpha=0.18, color='#e74c3c', label='Below base NAV')
    ax1.axhline(STARTING_NAV, color='grey', linestyle='--',
                linewidth=0.9, label=f'Base NAV = {STARTING_NAV}')
    ax1.annotate(f"Final NAV: {nav_v[-1]:.4f}",
                 xy=(tn[-1], nav_v[-1]),
                 xytext=(-70, 12), textcoords='offset points',
                 fontsize=9, color='#1a6496',
                 arrowprops=dict(arrowstyle='->', color='#1a6496', lw=0.8))
    ax1.set_title('BankNifty Short Strangle — Equity Curve (Base NAV = 100)',
                  fontsize=13, fontweight='bold', pad=10)
    ax1.set_xlabel('Trade Number', fontsize=10)
    ax1.set_ylabel('NAV', fontsize=10)
    ax1.legend(fontsize=9, loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    plt.tight_layout(); plt.show()
    buf1 = BytesIO()
    fig1.savefig(buf1, format='png', dpi=150, bbox_inches='tight')
    buf1.seek(0); plt.close(fig1)

    # ── Drawdown Curve ───────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(14, 4))
    ax2.fill_between(tn, dd_v, 0, color='#c0392b', alpha=0.55)
    ax2.plot(tn, dd_v, color='#c0392b', linewidth=1.3)
    ax2.axhline(0, color='grey', linestyle='--', linewidth=0.7)
    mi = int(np.argmin(dd_v))
    if dd_v[mi] < 0:
        ax2.annotate(
            f"Max DD: {dd_v[mi]:.2f}%",
            xy=(tn[mi], dd_v[mi]),
            xytext=(tn[mi] + max(len(tn)//15, 2), dd_v[mi] * 0.5),
            arrowprops=dict(arrowstyle='->', color='#7b241c', lw=0.9),
            fontsize=9, color='#7b241c', fontweight='bold'
        )
    ax2.set_title('BankNifty Short Strangle — Drawdown from Peak (%)',
                  fontsize=13, fontweight='bold', pad=10)
    ax2.set_xlabel('Trade Number', fontsize=10)
    ax2.set_ylabel('Drawdown %', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    plt.tight_layout(); plt.show()
    buf2 = BytesIO()
    fig2.savefig(buf2, format='png', dpi=150, bbox_inches='tight')
    buf2.seek(0); plt.close(fig2)

    STEP_TIMES['8_charts'] = time.time() - t0
    print(f"[STEP 8]  Charts rendered   Time: {STEP_TIMES['8_charts']:.2f}s")
    return buf1, buf2


# ════════════════════════════════════════════════════════════
#  EXCEL STYLE CONSTANTS
# ════════════════════════════════════════════════════════════
_NAVY = '1F497D'; _WHITE = 'FFFFFF'; _ALTROW = 'F2F7FC'
_SUBHDR = 'BDD7EE'; _GREEN = 'C6EFCE'; _RED = 'FFC7CE'; _YELLOW = 'FFEB9C'

H_FILL = PatternFill('solid', start_color=_NAVY)
A_FILL = PatternFill('solid', start_color=_ALTROW)
W_FILL = PatternFill('solid', start_color=_WHITE)
G_FILL = PatternFill('solid', start_color=_GREEN)
R_FILL = PatternFill('solid', start_color=_RED)
B_FILL = PatternFill('solid', start_color=_SUBHDR)
Y_FILL = PatternFill('solid', start_color=_YELLOW)
SL_FILL= PatternFill('solid', start_color='FFE0CC')

H_FONT = Font(bold=True, color=_WHITE,   name='Calibri', size=10)
B_FONT = Font(bold=True, color='000000', name='Calibri', size=10)
N_FONT = Font(name='Calibri', size=9)
BDR    = Border(left=Side(style='thin'), right=Side(style='thin'),
                top=Side(style='thin'),  bottom=Side(style='thin'))
CTR    = Alignment(horizontal='center', vertical='center', wrap_text=True)
LFT    = Alignment(horizontal='left',   vertical='top',    wrap_text=True)


def _hrow(ws, row, cols):
    for ci, val in enumerate(cols, 1):
        c = ws.cell(row=row, column=ci, value=val)
        c.font = H_FONT; c.fill = H_FILL; c.alignment = CTR; c.border = BDR


def _cell(ws, row, col, val, font=None, fill=None, align=None):
    c = ws.cell(row=row, column=col, value=val)
    c.font = font or N_FONT; c.fill = fill or W_FILL
    c.alignment = align or CTR; c.border = BDR
    return c


def _autowidth(ws, mn=11, mx=28):
    for col in ws.columns:
        w = max((len(str(c.value)) if c.value is not None else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(w+2, mn), mx)


# ════════════════════════════════════════════════════════════
#  STEP 9 — WRITE EXCEL WORKBOOK (4 sheets)
# ════════════════════════════════════════════════════════════
def write_excel(ts, stats, buf_eq, buf_dd, path, total_elapsed, step_times):
    t0 = time.time()
    print("[STEP 9]  Writing Excel workbook ...")
    wb = Workbook()

    # ══════════════════════════════════════════════════════════
    #  SHEET 1 — GUIDE
    # ══════════════════════════════════════════════════════════
    ws_g = wb.active
    ws_g.title = 'Guide'
    ws_g.sheet_view.showGridLines = False
    ws_g.column_dimensions['A'].width = 110

    guide = [
        ("BANKNIFTY SHORT STRANGLE BACKTEST — GUIDE & DOCUMENTATION", "title"),
        ("", "blank"),
        ("OBJECTIVE", "section"),
        ("Backtest a 09:20 short strangle trading system on BankNifty using one year "
         "of 1-minute options and index data (2023). The system sells one CE and one PE "
         "option each Week-1 trading day, selecting strikes whose premium is closest to "
         "Rs.50 at 09:20, and exits at 15:20 or on a 50% stop-loss.", "body"),
        ("", "blank"),
        ("STRATEGY OVERVIEW", "section"),
        ("A SHORT STRANGLE involves simultaneously SELLING a call (CE) and a put (PE) "
         "option with the same expiry but different strikes. The strategy profits from "
         "time decay when the underlying stays within a range. Since both options are "
         "sold, the trader RECEIVES premium upfront. Risk arises if BankNifty moves "
         "sharply, causing one leg to hit the stop-loss.", "body"),
        ("", "blank"),
        ("TASK BREAKDOWN", "section"),
        ("1. STRIKE SELECTION MODULE", "sub"),
        ("   At 09:20 each Week-1 trading day, scan all available CE and PE strikes. "
         "   Select the one strike per type whose 1-minute Close is closest to Rs.50. "
         "   This ensures near-the-money options with a defined target premium.", "body"),
        ("", "blank"),
        ("2. SIGNAL GENERATION MODULE", "sub"),
        (f"   ENTRY  : {ENTRY_TIME} candle close price", "body"),
        (f"   EXIT   : {EXIT_TIME} candle close price, OR stop-loss — whichever first", "body"),
        (f"   SL     : 50% of entry price. Check the High column of each 1-min bar.", "body"),
        ("   If any bar High >= Entry Price x 1.50 — exit at that SL price immediately.", "body"),
        ("   (We SOLD first — a rising option price is a loss for the short seller)", "body"),
        ("", "blank"),
        ("3. POSITION SIZING MODULE", "sub"),
        (f"   Fixed 1 lot per leg per day. Lot size = {LOT_SIZE}. Quantity = {QUANTITY} units.", "body"),
        ("   No compounding — same 1 lot regardless of running P&L.", "body"),
        ("", "blank"),
        ("4. UNIVERSE / TRADING SCHEDULE", "sub"),
        ("   Trade ONLY Week-1 days: from Monday to the first Wednesday of each month.", "body"),
        ("   The first Wednesday = weekly expiry day for BankNifty options.", "body"),
        ("   A trade is taken EVERY Week-1 day (both expiry and non-expiry days).", "body"),
        ("", "blank"),
        ("SHEETS IN THIS WORKBOOK", "section"),
        ("Guide       (this sheet) — Documentation, formulas, configuration, interpretation.", "body"),
        ("Tradesheet  — One row per option leg (CE + PE) per day. All required columns.", "body"),
        ("Statistics  — CAGR, Max Drawdown, Winners/Losers, Avg P&L, Monthly P&L, Timing.", "body"),
        ("Charts      — Equity curve (trade-wise NAV) and drawdown chart, high-resolution.", "body"),
        ("", "blank"),
        ("KEY FORMULAS", "section"),
        ("Entry Value         = Entry Price x Quantity", "body"),
        ("Exit Value          = Exit Price x Quantity", "body"),
        ("Gross P&L per leg   = (Entry Price - Exit Price) x Quantity", "body"),
        ("  Positive = profit (option price fell after we sold it)", "body"),
        ("  Negative = loss (option price rose after we sold it)", "body"),
        ("Cumulative P&L      = Running sum of Gross P&L trade-by-trade", "body"),
        ("Available Capital   = Starting Capital + Day-level Cumulative P&L", "body"),
        ("NAV (equity curve)  = 100 + (Cumulative Trade P&L / Starting Capital) x 100", "body"),
        ("CAGR                = (Final NAV / Starting NAV) ^ (1 / Years) - 1", "body"),
        ("  Years = Total Week-1 trading days / 252", "body"),
        ("Max Drawdown        = Largest % fall from any running NAV peak to subsequent trough", "body"),
        ("Monthly P&L %       = (End NAV this month / End NAV previous month - 1) x 100", "body"),
        ("Win Rate            = Count(Gross P&L > 0) / Total Trades x 100", "body"),
        ("", "blank"),
        ("CONFIGURATION USED", "section"),
        (f"Starting Capital    : Rs. {INITIAL_CAPITAL:,.0f}", "body"),
        (f"Base NAV            : {STARTING_NAV}", "body"),
        (f"Lot Size            : {LOT_SIZE}  |  Lots per day: {NUM_LOTS}  |  Quantity: {QUANTITY} units", "body"),
        (f"Target Premium      : Rs. {TARGET_PREM}", "body"),
        (f"Stop-Loss           : {int(SL_PCT*100)}% of entry price (detected via High column)", "body"),
        (f"Entry Time          : {ENTRY_TIME} 1-minute candle close", "body"),
        (f"Exit Time           : {EXIT_TIME} 1-minute candle close (if no SL triggered)", "body"),
        ("", "blank"),
        ("HOW TO READ THE TRADESHEET", "section"),
        ("- Each row = one option leg. Two rows per day (CE row + PE row).", "body"),
        ("- CE rows appear before PE rows for the same date.", "body"),
        ("- SL Hit = YES: stop-loss triggered before 15:20. Exit price = Entry x 1.50.", "body"),
        ("- Exit Time: actual time of exit (15:20, or the time of the SL-trigger bar).", "body"),
        ("- Gross P&L column is GREEN for profit, RED for loss.", "body"),
        ("- Cumulative P&L is trade-wise (not day-wise), matching the NAV equity curve.", "body"),
        ("- Available Capital is day-level (same value for both legs of the same day).", "body"),
        ("- BankNifty Spot Close is the 09:20 index close for that day.", "body"),
        ("- Is Expiry = YES: first Wednesday of month (weekly expiry day).", "body"),
        ("", "blank"),
        ("CODE TIMING (seconds)", "section"),
    ]
    for k, v in step_times.items():
        guide.append((f"   {k:<44} {v:.2f} s", "body"))
    guide += [
        ("", "blank"),
        (f"   {'TOTAL RUN TIME':<44} {total_elapsed:.1f} s", "sub"),
        ("", "blank"),
        ("   Target: < 60 seconds end-to-end (including all data loading and processing)", "body"),
    ]

    r = 1
    for text, kind in guide:
        c = ws_g.cell(row=r, column=1, value=text)
        if   kind == "title":
            c.font = Font(bold=True, size=14, name='Calibri', color=_NAVY)
        elif kind == "section":
            c.font = Font(bold=True, size=11, name='Calibri', color=_NAVY)
        elif kind == "sub":
            c.font = Font(bold=True, size=10, name='Calibri')
        else:
            c.font = Font(size=10, name='Calibri')
        c.alignment = LFT
        r += 1

    # ══════════════════════════════════════════════════════════
    #  SHEET 2 — TRADESHEET
    # ══════════════════════════════════════════════════════════
    ws_t = wb.create_sheet('Tradesheet')
    ws_t.sheet_view.showGridLines = False
    ws_t.row_dimensions[1].height = 28

    headers = [
        'Entry Date', 'Entry Time', 'Exit Date', 'Exit Time',
        'Option Ticker', 'Strike Price', 'Option Type',
        'Entry Price', 'Exit Price', 'SL Hit',
        'Quantity', 'Entry Value', 'Exit Value',
        'Gross P&L', 'Cumulative P&L', 'P&L %',
        'Available Capital', 'BankNifty Spot Close', 'Is Expiry', 'NAV'
    ]
    _hrow(ws_t, 1, headers)

    for ri, (_, row) in enumerate(ts.iterrows(), 2):
        pnl    = float(row['GrossPnL'])
        bg     = A_FILL if ri % 2 == 0 else W_FILL
        sl_hit = bool(row.get('SL_Hit', False))

        spot_val = row.get('Spot_Close')
        spot_out = round(float(spot_val), 2) if pd.notna(spot_val) and spot_val else ''

        row_vals = [
            str(row['Date']),
            ENTRY_TIME,
            str(row['Date']),
            str(row.get('ExitTime', EXIT_TIME)),
            str(row['EntryTicker']),
            int(row['Strike']),
            str(row['OptionType']),
            round(float(row['EntryPrice']), 2),
            round(float(row['ExitPrice']),  2),
            'YES' if sl_hit else 'NO',
            int(row['Quantity']),
            round(float(row['EntryValue']), 2),
            round(float(row['ExitValue']),  2),
            round(pnl, 2),
            round(float(row['CumPnL']), 2),
            round(float(row['PnL_Pct']), 4),
            round(float(row['AvailCap']), 2),
            spot_out,
            'YES' if bool(row['IsExpiry']) else 'NO',
            round(float(row['NAV']), 4),
        ]
        for ci, val in enumerate(row_vals, 1):
            if   ci == 14:  fill = G_FILL if pnl > 0 else (R_FILL if pnl < 0 else bg)
            elif ci == 10:  fill = SL_FILL if sl_hit else bg
            else:           fill = bg
            _cell(ws_t, ri, ci, val, fill=fill)

    ws_t.freeze_panes = 'A2'
    _autowidth(ws_t)

    # ══════════════════════════════════════════════════════════
    #  SHEET 3 — STATISTICS
    # ══════════════════════════════════════════════════════════
    ws_s = wb.create_sheet('Statistics')
    ws_s.sheet_view.showGridLines = False
    for col, w in zip('ABCD', [32, 18, 18, 18]):
        ws_s.column_dimensions[col].width = w

    def s_title(row, text):
        ws_s.merge_cells(start_row=row, start_column=1, end_row=row, end_column=4)
        c = ws_s.cell(row=row, column=1, value=text)
        c.font = Font(bold=True, color=_WHITE, name='Calibri', size=10)
        c.fill = PatternFill('solid', start_color=_NAVY)
        c.alignment = LFT; c.border = BDR
        ws_s.row_dimensions[row].height = 20

    def s_sub(row, cols):
        for ci, v in enumerate(cols, 1):
            c = ws_s.cell(row=row, column=ci, value=v)
            c.font = B_FONT; c.fill = B_FILL; c.alignment = CTR; c.border = BDR

    def s_row(row, cols, hl=None):
        for ci, v in enumerate(cols, 1):
            fill = Y_FILL if ci == hl else (A_FILL if row % 2 == 0 else W_FILL)
            _cell(ws_s, row, ci, v, fill=fill)

    r = 1

    # 1. Performance Summary
    s_title(r, "1.  PERFORMANCE SUMMARY"); r += 1
    s_sub(r, ["Metric", "Value", "", ""]); r += 1
    for label, val in [
        ("CAGR (%)",                     stats['CAGR_%']),
        ("Max Drawdown (%)",              stats['Max_Drawdown_%']),
        ("Total Week-1 Trading Days",     ts['Date'].nunique()),
        ("Total Option Legs Traded",      len(ts)),
        ("Starting Capital (Rs.)",        f"{INITIAL_CAPITAL:,}"),
        ("Starting NAV",                  STARTING_NAV),
        ("Ending NAV",                    round(float(ts['NAV'].iloc[-1]), 4)),
        ("Total Gross P&L (Rs.)",         round(float(ts['GrossPnL'].sum()), 2)),
    ]:
        s_row(r, [label, val, "", ""], hl=2); r += 1
    r += 1

    # 2. Winner / Loser Analysis
    s_title(r, "2.  WINNER / LOSER ANALYSIS"); r += 1
    s_sub(r, ["Metric", "CE (Calls)", "PE (Puts)", "Combined"]); r += 1
    for label, ck, pk, bk in [
        ("Number of Winners", 'CE_Winners',  'PE_Winners',  'Both_Winners'),
        ("Number of Losers",  'CE_Losers',   'PE_Losers',   'Both_Losers'),
        ("Total Trades",      'CE_Total',    'PE_Total',    'Both_Total'),
        ("Win %",             'CE_Win_Pct',  'PE_Win_Pct',  'Both_Win_Pct'),
        ("Loss %",            'CE_Loss_Pct', 'PE_Loss_Pct', 'Both_Loss_Pct'),
    ]:
        s_row(r, [label, stats[ck], stats[pk], stats[bk]]); r += 1
    r += 1

    # 3. Average % P&L
    s_title(r, "3.  AVERAGE % P&L  (Expiry Days vs Non-Expiry Days)"); r += 1
    s_sub(r, ["Day Type", "CE Avg P&L %", "PE Avg P&L %", "Combined Avg P&L %"]); r += 1
    for label, suffix in [("Expiry Days (Wednesday)", "Expiry"),
                           ("Non-Expiry Days (Mon / Tue)", "NonExpiry")]:
        s_row(r, [label,
                  stats.get(f'CE_{suffix}_Avg_PnL_%', ''),
                  stats.get(f'PE_{suffix}_Avg_PnL_%', ''),
                  stats.get(f'Both_{suffix}_Avg_PnL_%', '')]); r += 1
    r += 1

    # 4. Monthly P&L Table
    s_title(r, "4.  MONTHLY P&L TABLE  (NAV-Based)"); r += 1
    s_sub(r, ["Month", "End NAV", "Prev Month End NAV", "Monthly P&L %"]); r += 1
    for _, mr in stats['Monthly_PnL_Table'].reset_index().iterrows():
        pct   = float(mr['Monthly_PnL%'])
        pfill = G_FILL if pct > 0 else (R_FILL if pct < 0 else W_FILL)
        vals  = [str(mr['YM']),
                 round(float(mr['EndNAV']),  4),
                 round(float(mr['PrevNAV']), 4),
                 round(pct, 2)]
        for ci, val in enumerate(vals, 1):
            c = _cell(ws_s, r, ci, val,
                      fill=(A_FILL if r % 2 == 0 else W_FILL))
            if ci == 4:
                c.fill = pfill
        r += 1
    r += 1

    # 5. Code Timing
    s_title(r, "5.  CODE EXECUTION TIMING (seconds)"); r += 1
    s_sub(r, ["Step", "Time (s)", "", ""]); r += 1
    for sname, sec in step_times.items():
        s_row(r, [sname, round(sec, 2), "", ""]); r += 1
    s_row(r, ["TOTAL RUN TIME", round(total_elapsed, 2), "", ""], hl=2); r += 1
    s_row(r, ["Target",         "< 60 seconds", "", ""]); r += 1

    # ══════════════════════════════════════════════════════════
    #  SHEET 4 — CHARTS
    # ══════════════════════════════════════════════════════════
    ws_c = wb.create_sheet('Charts')
    ws_c.sheet_view.showGridLines = False
    hdr = ws_c.cell(row=1, column=1,
                    value='BankNifty Short Strangle — Equity Curve & Drawdown Chart')
    hdr.font = Font(bold=True, size=13, name='Calibri', color=_NAVY)
    ws_c.row_dimensions[1].height = 22

    ws_c.cell(row=2, column=1,
              value='Equity Curve (Base NAV = 100) — calculated trade-wise'
              ).font = Font(bold=True, size=10, name='Calibri', color='444444')

    img1 = XLImage(buf_eq); img1.width = 960; img1.height = 350
    ws_c.add_image(img1, 'A3')

    ws_c.cell(row=26, column=1,
              value='Drawdown from Peak (%) — annotated with maximum drawdown'
              ).font = Font(bold=True, size=10, name='Calibri', color='444444')

    img2 = XLImage(buf_dd); img2.width = 960; img2.height = 290
    ws_c.add_image(img2, 'A27')

    wb.save(path)
    elapsed = time.time() - t0
    STEP_TIMES['9_write_excel'] = elapsed
    print(f"          Saved -> {path}")
    print(f"          Time: {elapsed:.1f}s")


# ════════════════════════════════════════════════════════════
#  MAIN
# ════════════════════════════════════════════════════════════
def main():
    GRAND_START = time.time()
    print("=" * 62)
    print("  BANKNIFTY SHORT STRANGLE BACKTEST — FINAL SUBMISSION")
    print("=" * 62)

    df          = load_options(OPTIONS_PATH)
    spot        = load_spot(SPOT_PATH)
    week1_dates = get_week1_dates(df['Date'].unique())
    selected    = select_strikes(df, week1_dates)
    merged      = get_exits(df, selected)
    ts          = build_tradesheet(merged, spot, week1_dates)
    stats       = compute_stats(ts)
    buf_eq, buf_dd = make_charts(stats)

    grand_total = time.time() - GRAND_START
    write_excel(ts, stats, buf_eq, buf_dd,
                OUTPUT_PATH, grand_total, STEP_TIMES)

    try:
        from google.colab import files
        files.download(OUTPUT_PATH)
        print("\n  Excel file downloading to your computer ...")
    except ImportError:
        print(f"\n  File saved locally: {OUTPUT_PATH}")

    print("\n" + "=" * 62)
    print("  BACKTEST RESULTS SUMMARY")
    print("=" * 62)
    print(f"  Total Trades (legs)   : {len(ts)}")
    print(f"  Week-1 Trading Days   : {ts['Date'].nunique()}")
    print(f"  Starting Capital      : Rs. {INITIAL_CAPITAL:,}")
    print(f"  Total Gross P&L       : Rs. {ts['GrossPnL'].sum():,.2f}")
    print(f"  CAGR                  : {stats['CAGR_%']:.2f}%")
    print(f"  Max Drawdown          : {stats['Max_Drawdown_%']:.2f}%")
    print(f"  CE  Win Rate          : {stats['CE_Win_Pct']:.1f}%  "
          f"({stats['CE_Winners']}W / {stats['CE_Losers']}L)")
    print(f"  PE  Win Rate          : {stats['PE_Win_Pct']:.1f}%  "
          f"({stats['PE_Winners']}W / {stats['PE_Losers']}L)")
    print(f"  Combined Win Rate     : {stats['Both_Win_Pct']:.1f}%")
    print(f"  Starting NAV          : {STARTING_NAV}")
    print(f"  Ending NAV            : {round(float(ts['NAV'].iloc[-1]), 4)}")
    print(f"  Total Run Time        : {grand_total:.1f}s")
    print("=" * 62)
    print("\n  STEP-BY-STEP TIMING")
    print("  " + "-" * 50)
    for step, sec in STEP_TIMES.items():
        bar = '#' * int(sec / grand_total * 25)
        print(f"  {step:<36} {sec:>6.2f}s  {bar}")
    print(f"  {'TOTAL':<36} {grand_total:>6.1f}s")
    print("=" * 62)


if __name__ == '__main__':
    main()